# Salesforce to Databricks Pipeline Generation

This notebook generates Salesforce to Databricks ingestion pipelines using Databricks Asset Bundles.

## Process Overview
1. **Pipeline grouping**: Groups Salesforce objects by prefix + priority combinations
2. **YAML generation**: Creates Databricks Asset Bundle YAML file

## Key Differences from SQL Server
- **No Gateways**: Salesforce is a SaaS connector and does NOT require gateway configuration
- **Simpler Configuration**: Only requires connection name and schedule
- **Prefix + Priority Grouping**: Each unique (prefix, priority) pair becomes a separate pipeline

## Prerequisites
- Upload the `load_balancing` and `deployment` directories to your Databricks workspace
- Ensure you have a CSV file with your Salesforce objects
- Have a Databricks connection configured for Salesforce

## Install Required Libraries

Uncomment and run if needed:

In [0]:
# %pip install -r requirements.txt

## Import Required Modules

Install PyYAML (needed to import run_complete_pipeline_generation)

In [0]:
%pip install pyyaml

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules

In [0]:
import pandas as pd
import os

# Import the complete pipeline generation function
from run_pipeline_generation import run_complete_pipeline_generation

## Configure Parameters

**IMPORTANT**: Modify these parameters for your environment before running!

In [0]:
# Input CSV configuration
INPUT_CSV = 'load_balancing/examples/example_config.csv'  # Update with your CSV path

# Salesforce connection configuration
DEFAULT_CONNECTION_NAME = 'sfdc-pravin-dabs'  # Your Databricks Salesforce connection name
DEFAULT_SCHEDULE = '*/15 * * * *'            # Cron schedule (every 15 minutes)

# Output paths
OUTPUT_YAML = 'deployment/resources/sfdc_pipeline.yml'
OUTPUT_CONFIG = 'deployment/examples/generated_config.csv'

## Understanding the Input CSV

Your CSV should have the following columns:

**Required Columns:**
- `source_database`: Usually "Salesforce"
- `source_schema`: "standard" or "custom"
- `source_table_name`: Salesforce object name (e.g., "Account", "Contact")
- `target_catalog`: Target Databricks catalog
- `target_schema`: Target Databricks schema
- `target_table_name`: Destination table name
- `prefix`: Business unit or logical grouping (e.g., "business_unit1")
- `priority`: Priority level (e.g., "01", "02")

**Optional Columns:**
- `connection_name`: Salesforce connection name (uses default if not specified)
- `schedule`: Cron schedule (uses default if not specified)
- `include_columns`: Comma-separated list of columns to include
- `exclude_columns`: Comma-separated list of columns to exclude

**Pipeline Grouping Logic:**
- Objects are grouped by unique combinations of (prefix, priority)
- Each group becomes a separate pipeline
- Example: All objects with prefix="business_unit1" and priority="01" go into one pipeline

## Example: Basic Pipeline Generation

This example uses the provided example CSV with default settings.

In [0]:
%sql
-- create catalog tapworks_catalog;
-- create schema tapworks_catalog.sfdc_test;

In [0]:
result_df = run_complete_pipeline_generation(
    input_csv='load_balancing/examples/tapworks_sfdc_config.csv',
    default_connection_name='sfdc-pravin-dabs',
    default_schedule='*/15 * * * *',
    output_yaml='/Workspace/Users/yas.mokri@databricks.com/lakehouse-tapworks/sfdc/dab_deployment/resources/sfdc_pipeline.yml',
    output_config='/Workspace/Users/yas.mokri@databricks.com/lakehouse-tapworks/sfdc/dab_deployment/resources/generated_config.csv'
)




In [0]:
result_df = run_complete_pipeline_generation(
    input_csv='load_balancing/examples/example_config.csv',
    default_connection_name='sfdc_connection',
    default_schedule='*/15 * * * *',
    output_yaml='deployment/resources/sfdc_pipeline.yml',
    output_config='deployment/examples/generated_config.csv'
)

## Example: Custom Configuration

This example shows how to use custom parameters.

In [0]:
result_df = run_complete_pipeline_generation(
    input_csv=INPUT_CSV,
    default_connection_name=DEFAULT_CONNECTION_NAME,
    default_schedule=DEFAULT_SCHEDULE,
    output_yaml=OUTPUT_YAML,
    output_config=OUTPUT_CONFIG
)

## Review Generated Configuration

Display the generated pipeline configuration:

In [0]:
display(result_df)

## Summary Statistics

In [0]:
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total Salesforce objects: {len(result_df)}")
print(f"Total pipelines: {result_df['pipeline_group'].nunique()}")
print(f"\nObjects per pipeline group:")
print(result_df.groupby('pipeline_group').size())
print(f"\nBreakdown by prefix and priority:")
print(result_df.groupby(['prefix', 'priority']).size())
print("="*80)

## View Pipeline Groups

See which objects are in each pipeline:

In [0]:
for pipeline_group in sorted(result_df['pipeline_group'].unique()):
    group_df = result_df[result_df['pipeline_group'] == pipeline_group]
    prefix = group_df['prefix'].iloc[0]
    priority = group_df['priority'].iloc[0]
    print(f"\nPipeline Group {pipeline_group} ({prefix} - Priority {priority}):")
    print("  Objects:")
    for _, row in group_df.iterrows():
        print(f"    - {row['source_table_name']} → {row['target_catalog']}.{row['target_schema']}.{row['target_table_name']}")

## View Generated YAML

Display the generated YAML file content:

In [0]:
with open(OUTPUT_YAML, 'r') as f:
    print(f.read())

## Next Steps

1. **Review Generated Files**: Check the YAML file and configuration CSV
2. **Verify Connection**: Ensure your Salesforce connection exists in Databricks:
   - Go to Databricks Workspace → Connections
   - Verify connection name matches `DEFAULT_CONNECTION_NAME`
3. **Deploy Using Databricks Asset Bundles**:
   ```bash
   cd deployment
   databricks bundle validate -t dev
   databricks bundle deploy -t dev
   ```
4. **Monitor**: Check your Databricks workspace for the deployed pipelines

## Troubleshooting

**Common Issues:**

1. **Connection not found**: Ensure the Salesforce connection is created in Databricks
2. **Import errors**: Make sure all files are uploaded to the workspace
3. **Pipeline conflicts**: Check for duplicate prefix + priority combinations
4. **CSV format errors**: Verify all required columns are present

**Validation:**

```python
# Check for missing required columns
required_cols = ['source_database', 'source_schema', 'source_table_name', 
                'target_catalog', 'target_schema', 'target_table_name',
                'prefix', 'priority']
input_df = pd.read_csv(INPUT_CSV)
missing = [col for col in required_cols if col not in input_df.columns]
if missing:
    print(f"❌ Missing required columns: {missing}")
else:
    print("✓ All required columns present")
```